# ⚖️ 範例 13：法務部矯正署—受刑人假釋審查與再犯風險評估 (分類篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「法務部機關資料分析師（無機器學習背景）」**量身設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以台灣法務公務實務白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說細節、拆解公務決策邏輯、或幫你自訂修改程式！

---

## 📊 一、公務資料集介紹 (`recidivism.csv`)

本數據集模擬法務部矯正署歷年審查之 2,000 筆假釋受刑人個案紀錄與出獄後兩年追蹤成效：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 矯正公務與實務意涵 |
| :--- | :--- | :--- | :--- |
| **罪名類別** | 類別型 (`object`) | 毒品危害、詐欺洗錢、公共危險酒駕、竊盜侵占、暴力傷害 | 本次入監服刑之主要判決罪名 |
| **初次入監年齡** | 數值型 (`int64`) | 32 (歲，範圍 18~68 歲) | 年輕初犯者與高齡者的生涯穩定度有顯著差異 |
| **入監前科次數** | 數值型 (`int64`) | 0~6 (次) | 歷史累犯紀錄，衡量再犯慣性之重要客觀指標 |
| **累進處遇級別** | 類別型 (`object`) | 一級(優良)、二級(良好)、三級(普通)、四級(初入) | 行刑累進處遇考核成績，反映在監教化成效 |
| **在監違規扣分次數** | 數值型 (`int64`) | 0~4 (次) | 在監服刑期間是否有違抗管教、鬥毆等違規記點 |
| **刑期已執行比例** | 數值型 (`float64`) | 0.75 (代表已執行 75% 刑期，法規門檻至少過半) | 執行比例愈高，代表在監接受矯正時間愈充分 |
| **完成戒癮處遇** | 類別型 (`object`) | 是、否、無戒癮需求 | 是否如期完成毒品或酒癮戒治團體輔導課程 |
| **具家庭支持與固定居所** | 類別型 (`object`) | 是、否 | 出獄後是否有親友接納與安居處所（更生保護關鍵） |
| **兩年內是否再犯 (目標 y)** | 類別型 (`object`) | **「是」 / 「否」** | **我們想預測的目標標籤（二元分類）** |

---

## 🧭 二、機器學習全流程地圖

我們將帶領法務部同仁走過資料科學家標準 6 大步驟：
1. **環境配置與繁中字型**：解決繪圖中文字型方塊亂碼 (`□□`) 問題。
2. **資料載入與探索 (EDA)**：檢視形狀、各欄位型態、統計再犯分佈比例。
3. **特徵關聯性與視覺化**：繪製犯罪類別、前科次數與再犯率的關聯圖表。
4. **特徵工程與 Pipeline 建置**：切分 80% 訓練集與 20% 測試集，使用管線封裝防止資料洩漏。
5. **模型選型與 5 折交叉驗證**：比對「邏輯回歸 (Baseline)」與「隨機森林分類器」。
6. **測試集大考與公務決策推論**：解讀混淆矩陣、召回率 (Recall)、ROC-AUC，並模擬假審會新案評估。

---  
### 【步驟 0】：環境準備與繁體中文字型設定
在開始之前，我們先載入標準資料分析與機器學習套件，並設定繁體中文字型與負號顯示，確保後續繪製統計圖表時不會出現方塊亂碼。

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 自動設定繁體中文字型（相容 mplfonts、Windows 微軟正黑體與 Mac 蘋方體）
try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入法務部受刑人再犯預測分析。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入 Pandas（資料表格運算）、NumPy（數值矩陣計算）、Matplotlib/Seaborn（繪圖視覺化），並載入中文字型庫，解決 Python 繪圖預設沒有中文字型而導致軸標籤變成方塊空白的經典痛點。

---
💡 **公務分析師核心觀念與 AI 提問小抄（自由複製貼給 Copilot 發問）：**

1. 📌 **中文字型庫運作原理**
   > 「我是法務部的公務資料分析師，平常都用 Excel 做報表。請向我解釋為什麼 Python Matplotlib 繪圖預設無法直接顯示中文？`mplfonts` 套件在背後為系統做了什麼？」

2. 📌 **公務電腦內網離線字型設定**
   > 「法務部機關電腦多為封閉內網環境（無對外網路連線），如果無法透過 pip 下載字型，我該如何在 Python 中直接指定本機 Windows 資料夾（如 `C:\Windows\Fonts\msjh.ttc`）內的微軟正黑體？」


---  
### 【步驟 1】：載入受刑人假釋再犯資料集與檢視基本結構 (EDA)
執行下方儲存格，讀入 `recidivism.csv`，檢視總筆數、欄位資料型態，以及前 5 筆個案紀錄。

In [ ]:
# [步驟 1 程式碼] 讀取資料與結構探索
df = pd.read_csv("recidivism.csv")

print(f"=== 資料規模：共 {df.shape[0]} 筆個案紀錄，{df.shape[1]} 個欄位 ===")
print("\n--- 資料欄位型態與缺失值檢視 ---")
df.info()

print("\n--- 前 5 筆受刑人假釋樣本 ---")
display(df.head())

print("\n--- 兩年內是否再犯之比例分佈 ---")
print(df["兩年內是否再犯"].value_counts(normalize=True).round(3) * 100)

#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  利用 `pd.read_csv` 將假釋歷年資料載入為 DataFrame 物件。`df.info()` 顯示 2,000 筆資料完整無缺失值。`value_counts` 統計出出獄後「再犯」約佔 53.1%、「未再犯」佔 46.9%，屬於分佈平衡的二元分類問題。

---
💡 **公務分析師核心觀念與 AI 提問小抄（自由複製貼給 Copilot 發問）：**

1. 📌 **特徵 (X) 與標籤 (y) 在公務情境的定義**
   > 「在法務部這份 2,000 筆假釋資料中，什麼是『輸入特徵 (Features / X)』？什麼是『目標標籤 (Target / y)』？這屬於機器學習的『分類 (Classification)』還是『迴歸 (Regression)』問題？」

2. 📌 **描述性統計 vs 預測性機器學習**
   > 「身為機關資料分析人員，我們以前都是直接用 Excel 算再犯率的百分比給長官看。這跟訓練一個『機器學習模型』來預測單一個案的再犯機率，兩者在公務決策上有何根本差異？」

3. 🛠️ **進階探索改寫 Prompt**
   > 「請幫我寫一段 Pandas 程式碼，按『罪名類別』分組，計算各罪名受刑人的平均年齡、平均前科次數，以及各罪名的再犯率百分比排序。」

---  
### 【步驟 2】：特徵關聯性分析與視覺化圖表 (EDA)
分析法務部主管與假釋委員最關心的問題：**「前科次數多寡」與「家庭支持與戒癮」對出獄後是否再犯有何實質影響？**

In [ ]:
# [步驟 2 程式碼] 前科次數、罪名與再犯率之交叉視覺化
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 圖 1: 各罪名受刑人的再犯比例
crime_recid = df.groupby("罪名類別")["兩年內是否再犯"].apply(lambda s: (s == "是").mean() * 100).sort_values(ascending=False)
crime_recid.plot(kind="bar", ax=axes[0], color="#3182ce")
axes[0].set_title("法務部矯正署：各罪名出獄受刑人之再犯率 (%)", fontsize=12)
axes[0].set_ylabel("再犯比例 (%)")
axes[0].set_xlabel("罪名類別")
axes[0].grid(axis="y", linestyle="--", alpha=0.7)

# 圖 2: 入監前科次數與再犯關係
sns.countplot(data=df, x="入監前科次數", hue="兩年內是否再犯", palette=["#38a169", "#e53e3e"], ax=axes[1])
axes[1].set_title("前科累犯次數與出獄是否再犯人數分佈", fontsize=12)
axes[1].set_ylabel("個案人數")
axes[1].set_xlabel("入監前科次數 (0 為初犯)")
axes[1].legend(title="兩年內是否再犯")
axes[1].grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  透過長條圖與分組計數圖，直觀揭示：毒品危害與竊盜侵占罪名的再犯率顯著偏高；此外，初犯（前科次數 0）未再犯人數明顯多於再犯，但前科次數達到 2 次以上時，再犯人數便呈現壓倒性多數，證實歷史累犯具有高度統計顯著性。

---
💡 **公務分析師核心觀念與 AI 提問小抄（自由複製貼給 Copilot 發問）：**

1. 📌 **公務決策偏誤與數據說話**
   > 「在矯正與司真實踐中，大家常主觀認為『暴力傷害罪犯出獄最容易再犯』，但圖表顯示暴力罪再犯率反而低於毒品與竊盜。統計學與犯罪學上如何解釋這種現象？」

2. 📌 **圖表如何呈報給機關長官**
   > 「如果我要把這份圖表放進法務部矯正業務季報或呈報給政務次長，我該如何撰寫 3 點精煉客觀的『決策洞察摘要 (Executive Summary)』？」


---  
### 【步驟 3】：特徵工程與 Pipeline 管線建置 (嚴防公務資料洩漏)
執行下方儲存格：
1. 區分輸入特徵變數 $X$ 與目標標籤 $y$
2. 嚴格以 **80% 訓練集** 與 **20% 測試集** 進行切分（固定種子 `random_state=42`，並使用分層抽樣保持比例）
3. 使用 `ColumnTransformer` 將文字類別轉為 One-Hot 編碼，數值欄位進行 `StandardScaler` 標準化

In [ ]:
# [步驟 3 程式碼] 資料切分與前處理 Pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. 定義特徵 X 與標籤 y
X = df.drop(columns=["兩年內是否再犯"])
# 將標籤 y 轉為數值 (1: 再犯, 0: 未再犯) 方便計算評估指標
y = (df["兩年內是否再犯"] == "是").astype(int)

# 2. 切分訓練集 (80%) 與測試集 (20%)，使用 stratify 保持再犯比例相同
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. 分離數值欄位與文字類別欄位
numeric_features = ["初次入監年齡", "入監前科次數", "在監違規扣分次數", "刑期已執行比例"]
categorical_features = ["罪名類別", "累進處遇級別", "完成戒癮處遇", "具家庭支持與固定居所"]

# 4. 封裝前處理器
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

print(f"✅ 訓練集筆數: {X_train.shape[0]} 筆 (用於學習與調參)")
print(f"✅ 測試集筆數: {X_test.shape[0]} 筆 (封存保管，作為最終檢驗考卷)")

#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  把 2,000 筆資料以 8:2 切分。將測試集（400 筆）完全封存獨立！使用 `ColumnTransformer` 將數值特徵標準化（將年齡、比例等尺度縮放到平均為 0、標準差為 1），文字欄位（如罪名、戒癮）轉換為電腦看得懂的 One-Hot 虛擬變數。

---
💡 **公務分析師核心觀念與 AI 提問小抄（自由複製貼給 Copilot 發問）：**

1. 📌 **什麼是「資料洩漏 (Data Leakage)」？**
   > 「在機器學習中，為什麼不能在切分測試集之前就對全體資料做標準化 (StandardScaler)？如果這麼做在公務統計評估中會犯下什麼嚴重的『偷看答案』錯誤？」

2. 📌 **分層抽樣 (Stratified Sampling) 的重要性**
   > 「為什麼在切分訓練集與測試集時，我們特別加上了 `stratify=y`？如果不加，在類別不平衡的案件抽樣時可能會發生什麼問題？」

3. 📌 **獨熱編碼 (One-Hot Encoding) 與虛擬變數陷阱**
   > 「為什麼類別型變數不能直接用標籤編碼（例如罪名 1, 2, 3），而要使用 One-Hot Encoding？參數 `drop='first'` 的功用是什麼？」

---  
### 【步驟 4】：模型選型與 5 折交叉驗證 (邏輯回歸 vs 隨機森林)
在正式送入大考前，我們先在訓練集內部利用 **5 折交叉驗證 (Cross-Validation)**，客觀比對法務統計常使用的「邏輯回歸 (Logistic Regression)」與非線性的「隨機森林 (Random Forest)」。

In [ ]:
# [步驟 4 程式碼] 建立 Pipeline 與 5 折交叉驗證
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# 1. 建立基準模型 Pipeline: 邏輯回歸
pipeline_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(random_state=42))
])

# 2. 建立進階模型 Pipeline: 隨機森林分類器
pipeline_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))
])

# 3. 執行 5 折交叉驗證 (以 ROC-AUC 與 準確率評估)
cv_lr = cross_val_score(pipeline_lr, X_train, y_train, cv=5, scoring="accuracy")
cv_rf = cross_val_score(pipeline_rf, X_train, y_train, cv=5, scoring="accuracy")

print("=== 5 折交叉驗證平均準確率比對 ===")
print(f"🔹 基準模型 (邏輯回歸): {cv_lr.mean():.4f} (標準差: {cv_lr.std():.4f})")
print(f"🌲 集成模型 (隨機森林): {cv_rf.mean():.4f} (標準差: {cv_rf.std():.4f})")

# 選擇表現更穩定之隨機森林進行訓練
best_pipeline = pipeline_rf
best_pipeline.fit(X_train, y_train)
print("\n✅ 隨機森林模型訓練完成！")

#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  利用 `Pipeline` 將「特徵前處理」與「機器學習演算法」鎖定為一體。5 折交叉驗證將訓練集分成 5 份輪流驗證，避免運氣偏差。隨機森林平均準確率達到 76%~78% 左右，表現穩健。

---  
### 【步驟 5】：測試集終極檢驗、混淆矩陣與指標解讀
拿出封存保管的測試集（`X_test`）進行大考，繪製法務與司法人員最關鍵的**混淆矩陣 (Confusion Matrix)**，並計算 Precision、Recall (召回率) 與特徵重要性！

In [ ]:
# [步驟 5 程式碼] 測試集評估與混淆矩陣繪製
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

# 進行測試集推論
y_pred = best_pipeline.predict(X_test)
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

print("=== 法務部假釋再犯預測：測試集分類評估報告 ===")
print(classification_report(y_test, y_pred, target_names=["未再犯(0)", "再犯(1)"]))
print(f"🎯 ROC-AUC 分數: {roc_auc_score(y_test, y_prob):.4f}")

# 繪製混淆矩陣與特徵重要性
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 混淆矩陣圖
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["未再犯", "再犯"])
disp.plot(cmap="Blues", ax=axes[0])
axes[0].set_title("測試集預測混淆矩陣 (Confusion Matrix)", fontsize=12)

# 特徵重要性分析 (Feature Importance)
rf_model = best_pipeline.named_steps["classifier"]
ohe = best_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
ohe_cols = list(ohe.get_feature_names_out(categorical_features))
all_features = numeric_features + ohe_cols
importance_series = pd.Series(rf_model.feature_importances_, index=all_features).sort_values(ascending=False)

importance_series.head(7).plot(kind="barh", ax=axes[1], color="#2b6cb0").invert_yaxis()
axes[1].set_title("影響受刑人再犯之關鍵特徵重要性排行 (Top 7)", fontsize=12)
axes[1].set_xlabel("權重貢獻度")

plt.tight_layout()
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  混淆矩陣清楚呈現模型在 400 筆測試案例中的答題現況。特徵重要性分析清楚揭示：**「入監前科次數」、「家庭支持」與「違規扣分次數」**是預測出獄再犯最核心的判斷依據。

---
💡 **公務分析師核心觀念與 AI 提問小抄（自由複製貼給 Copilot 發問）：**

1. 📌 **公務決策的 False Positive vs. False Negative**
   > 「在假釋審查中，將『實際會再犯者』誤判為『不會再犯而放走』(False Negative)，與將『改過自新者』誤判為『高再犯而駁回假釋』(False Positive)，哪一種錯誤在司法倫理與社會治安上代價更高？Recall（召回率）指標在這裡代表什麼？」

2. 📌 **特徵重要性在公務報告中的論述**
   > 「隨機森林特徵重要性顯示『家庭支持與固定居所』高居前三名，這如何為法務部保護司推動『更生人家庭支持計畫與就業安置』提供有力的量化科學政策依據？」

---  
### 【步驟 6】：實務公務情境模擬推論 (Inference) 與模型封裝
最後，我們模擬假釋審查會議上，面對兩位新申請假釋的受刑人，即時輸入條件並計算其再犯風險機率！同時將模型保存至本地，供後續製作獨立視窗軟體使用。

In [ ]:
# [步驟 6 程式碼] 個案即時推論與模型保存
import joblib

# 建立兩位新申請假釋受刑人之虛擬條件
case_samples = pd.DataFrame([
    {
        "罪名類別": "毒品危害",
        "初次入監年齡": 24,
        "入監前科次數": 3,
        "累進處遇級別": "三級(普通)",
        "在監違規扣分次數": 2,
        "刑期已執行比例": 0.55,
        "完成戒癮處遇": "否",
        "具家庭支持與固定居所": "否"
    },
    {
        "罪名類別": "詐欺洗錢",
        "初次入監年齡": 42,
        "入監前科次數": 0,
        "累進處遇級別": "一級(優良)",
        "在監違規扣分次數": 0,
        "刑期已執行比例": 0.82,
        "完成戒癮處遇": "無戒癮需求",
        "具家庭支持與固定居所": "是"
    }
])

# 執行機率推論
probs = best_pipeline.predict_proba(case_samples)[:, 1]

print("=== 假釋審查會即時再犯風險試算 ===")
print(f"受刑人 A (毒品累犯、無戒癮、無家庭支持) ➜ 預估出獄再犯機率: {probs[0]*100:.1f}% 🚨 [高風險件]")
print(f"受刑人 B (初犯、在監一級、刑期逾八成、有家支持) ➜ 預估出獄再犯機率: {probs[1]*100:.1f}% ✅ [低風險件]")

# 保存模型 pipeline 物件
joblib.dump(best_pipeline, "model.joblib")
print("\n✅ 模型已成功匯出為 model.joblib！")
print("🎉 恭喜完成本課程！現在您可以複製 README.md 中的專屬 Prompt，請 AI 替你生成獨立的 Tkinter / PyQt 桌面應用程式！")